In [13]:
from pathlib import Path

import optuna
import numpy as np
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

from CustomSpeachDataset import CustomSpeachDataset
from SpeachClassifierModel import SpeachClassifierModel

In [14]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [15]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset"), preload=True)
dataset.to(device)

Preloading dataset from disk...


In [16]:
train_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_indices]
train_dataset = torch.utils.data.Subset(dataset, train_indices)
train_sample_weights = all_sample_weights[train_indices]
len(train_dataset), len(y_train)

(16447, 16447)

In [17]:
def train_new_model(writer: SummaryWriter, fold: int, params: dict, epochs: int, train_loader: DataLoader, val_loader: DataLoader) -> SpeachClassifierModel:
    model = SpeachClassifierModel()
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    loss_fn = nn.CrossEntropyLoss()

    best_val_loss = 99999999999.9

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                y_pred = model(data)
                loss = loss_fn(y_pred, target)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

        model.best_val_loss = best_val_loss

        writer.add_scalar(f"Loss/train/fold{fold}", train_loss, epoch)
        writer.add_scalar(f"Loss/val/fold{fold}", val_loss, epoch)

    return model

In [18]:
def train_models_5cv(writer: SummaryWriter, params: dict, skf: StratifiedKFold, epochs: int) -> float:
    total_val_loss = 0.0

    for fold, (train_index, val_index) in enumerate(skf.split(y_train, y_train)):
        train_sampler = WeightedRandomSampler(
            weights=train_sample_weights[train_index],
            num_samples=len(train_sample_weights[train_index]),
            replacement=True,
        )
        train_loader = DataLoader(train_dataset, batch_size=512, sampler=train_sampler)

        val_sampler = WeightedRandomSampler(
            weights=train_sample_weights[val_index],
            num_samples=len(train_sample_weights[val_index]),
            replacement=True,
        )
        val_loader = DataLoader(train_dataset, batch_size=1024, sampler=val_sampler)

        model = train_new_model(writer, fold, params, epochs, train_loader, val_loader)
        total_val_loss += model.best_val_loss

    avl_val_loss = total_val_loss / skf.n_splits
    return avl_val_loss


In [19]:
def objective_train(trial: optuna.trial.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-7, 1e-4, log=True)

    current_params = trial.params

    # Fewer epochs for hyperparameters optimization process
    epochs = 30
    skf = StratifiedKFold(n_splits=5, shuffle=True)

    with SummaryWriter(f"runs/trial_{trial.number}/folds") as writer:
        avg_val_loss = train_models_5cv(writer, current_params, skf, epochs)

    metrics = {
        "hparams/avg_val_loss": avg_val_loss
    }

    with SummaryWriter(f"runs/optimization") as writer:
        writer.add_hparams(current_params, metrics, run_name=f"trial_{trial.number}")
        writer.add_scalar("Loss/avg_val_loss", avg_val_loss, trial.number)

    return avg_val_loss

In [20]:
study = optuna.create_study(study_name="Study", direction='minimize')

[I 2026-05-14 22:53:10,573] A new study created in memory with name: Study


In [ ]:
study.optimize(objective_train, n_trials=10)